## VoxCPM2 AstrBot 语音克隆服务端 (固定音色持久化版)
- **首次设定或更换音色**：运行「单元格 0」上传参考音频与文字稿到 Google Drive。
- **每次重连 GPU**：按顺序运行 **单元格 1 → 单元格 2 → 单元格 3**，将生成的 `/v1` 地址通过 `/vox_url` 发给机器人或填入 AstrBot 后台。

In [ ]:
# 【单元格 0】仅首次设定或更换音色时运行（平时跳过）
import os
from google.colab import drive, files

drive.mount('/content/drive')
cache_dir = "/content/drive/MyDrive/VoxCPM2_Cache"
os.makedirs(cache_dir, exist_ok=True)

print("请上传你要设定的参考音频（会自动保存为 speaker.wav）：")
uploaded = files.upload()
if uploaded:
    src = next(iter(uploaded))
    os.replace(src, os.path.join(cache_dir, "speaker.wav"))
    print("✅ 参考音频已保存！")

# 根据你的实际参考音频内容修改下方文字稿（如不使用极致克隆可留空 ""）
prompt_text = "是对比同类型主播的各项数据，从10月19日起变化的观众数量和粉丝增长"
with open(os.path.join(cache_dir, "speaker_prompt.txt"), "w", encoding="utf-8") as f:
    f.write(prompt_text.strip())
print("✅ 参考文字稿已保存！")

In [ ]:
# 【单元格 1】检查 GPU 并安装依赖与 cloudflared
!nvidia-smi
!pip install -q voxcpm soundfile fastapi uvicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 【单元格 2】挂载云盘、校验固定音色并加载模型到本地高速盘
import os, glob, shutil, torch
from google.colab import drive

drive.mount('/content/drive')
DRIVE_CACHE = '/content/drive/MyDrive/VoxCPM2_Cache'
DRIVE_WAV = os.path.join(DRIVE_CACHE, 'speaker.wav')
DRIVE_TXT = os.path.join(DRIVE_CACHE, 'speaker_prompt.txt')

assert os.path.exists(DRIVE_WAV), f"❌ 未在云盘找到 {DRIVE_WAV}，请先运行单元格 0 上传音色！"

LOCAL_WAV = '/content/fixed_speaker.wav'
shutil.copy(DRIVE_WAV, LOCAL_WAV)

FIXED_PROMPT_TEXT = ""
if os.path.exists(DRIVE_TXT):
    with open(DRIVE_TXT, 'r', encoding='utf-8') as f:
        FIXED_PROMPT_TEXT = f.read().strip()

print(f"🔒 已锁定固定音色：{DRIVE_WAV}")
print(f"📝 已锁定参考文字稿：{FIXED_PROMPT_TEXT if FIXED_PROMPT_TEXT else '(空)'}")

local_hf_cache = '/content/VoxCPM2_LocalCache'
os.makedirs(local_hf_cache, exist_ok=True)
os.environ['HF_HOME'] = local_hf_cache
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(local_hf_cache, 'hub')
os.environ['HF_HUB_DISABLE_XET'] = '1'

_orig_load = torch.load
def _compat_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_load(*args, **kwargs)
torch.load = _compat_load

from huggingface_hub import snapshot_download
from voxcpm import VoxCPM

model_dir = snapshot_download(
    repo_id="openbmb/VoxCPM2",
    cache_dir=os.path.join(local_hf_cache, 'hub'),
)
model = VoxCPM.from_pretrained(
    model_dir,
    load_denoiser=False,
    optimize=False,
)
torch.load = _orig_load

# 预热推理，消除首条消息冷启动延迟
_ = model.generate(
    text="你好。",
    reference_wav_path=LOCAL_WAV,
    prompt_wav_path=LOCAL_WAV if FIXED_PROMPT_TEXT else None,
    prompt_text=FIXED_PROMPT_TEXT if FIXED_PROMPT_TEXT else None,
    cfg_value=2.0,
    inference_timesteps=6,
)
print("✅ 模型加载与音色预热完成！")

In [ ]:
# 【单元格 3】启动 OpenAI 兼容 TTS 服务并输出公网地址
import io, time, threading, subprocess, re
import soundfile as sf
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
import uvicorn

app = FastAPI()

class TTSRequest(BaseModel):
    input: str
    voice: str = "default"
    model: str = "voxcpm2"
    response_format: str = "wav"

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": "voxcpm2", "object": "model"}]}

@app.post("/v1/audio/speech")
def openai_compatible_tts(req: TTSRequest):
    kwargs = {
        "text": req.input.strip(),
        "reference_wav_path": LOCAL_WAV,
        "cfg_value": 2.0,
        "inference_timesteps": 7,
    }
    if FIXED_PROMPT_TEXT:
        kwargs["prompt_wav_path"] = LOCAL_WAV
        kwargs["prompt_text"] = FIXED_PROMPT_TEXT

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    return Response(content=buf.getvalue(), media_type="audio/wav")

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True
).start()
time.sleep(2)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("==================================================")
        print("🚀 复制下方地址发给机器人（/vox_url）或填入 AstrBot：")
        print(m.group(0) + "/v1")
        print("==================================================")
        break